# 第 30 章　結果の報告

配合と指定材齢から，測る前に強度を見積もる方法を確かめました。同じ結果を，次の二人に伝えます。

- 方法をこの先も試す候補に残すか考える人。
- 外れの原因を調べるため，次にどの試験記録を集めるか考える人。

前半では，同じ分析を二つの図と報告文へ組み直します。後半では，自分の第 27〜29 章の作業ノートから，読者が判断するために使う表や図を選んで改稿します。そのノートを手元に用意してください。

ローカルでは，このノートの近くまたは教材の `data` フォルダーにある `concrete-strength.csv` を読み込みます。Colab などで手元に見つからない場合は配布先から読み込みます。

## 計算順の下書き

> コンクリートの CSV を読み，学習 631 行と検証 208 行を取り出した。直線と 200 本の森を学習し，検証の MAE は直線 8.11，森 3.97 MPa だった。森を選び，学習平均の基準とともに保留 191 行へ適用すると，基準 13.43，森 5.55 MPa だった。保留を三つの材齢範囲に分けると，森は 4.00，4.67，8.37 MPa だった。

この記録からは，作業の順番が分かります。一文だけ先に渡すとしたら，二人の読み手に，どの結果を選びますか。図を出す前に，それぞれが最初に知りたいことを一つずつ言葉にします。

## 全体と条件を読み直す

第 29 章と同じ学習 631 行で森を学び直します。基準は学習表の平均強度を全行に返します。検証と保留の二つの表に同じ規則を使い，それぞれの行で絶対誤差を計算します。

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor

data_url = (
    "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials"
    "/main/data/concrete-strength.csv"
)
local_paths = [
    Path("concrete-strength.csv"), Path("data/concrete-strength.csv"),
    Path("../data/concrete-strength.csv"),
]
data_source = next((path for path in local_paths if path.is_file()), data_url)
concrete = pd.read_csv(data_source)
train = concrete.loc[concrete["split"] == "train"]
valid = concrete.loc[concrete["split"] == "validation"]
held = concrete.loc[concrete["split"] == "holdout"]
features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]

forest = RandomForestRegressor(
    n_estimators=200, max_features=0.7, random_state=2026, n_jobs=2,
)
forest.fit(train[features], train["strength"])
training_mean = train["strength"].mean()


def make_evaluation_table(part):
    scored = part[["record_id", "recipe_id", "age", "strength"]].copy()
    scored["forest_prediction"] = forest.predict(part[features])
    scored["baseline_error"] = (scored["strength"] - training_mean).abs()
    scored["forest_error"] = (scored["strength"] - scored["forest_prediction"]).abs()
    return scored


valid_scored = make_evaluation_table(valid)
held_scored = make_evaluation_table(held)

### 方法を候補に残すか

まず，保留と検証を別の段に表示します。同じ段の二点は同じ記録に対する基準と森の MAE です。固定後にも基準より誤差が小さかったかと聞かれたら，どの二点を比べますか。

In [ ]:
summary = pd.DataFrame(
    [
        {
            "rows": len(part),
            "recipes": part["recipe_id"].nunique(),
            "baseline_mae": part["baseline_error"].mean(),
            "forest_mae": part["forest_error"].mean(),
        }
        for part in [held_scored, valid_scored]
    ],
    index=["Held-out", "Validation"],
)
summary.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.9, 3.65), layout="constrained")
for y, (label, values) in zip([1, 0], summary.iterrows()):
    color = "#245e7d" if label == "Held-out" else "#697b86"
    baseline = values["baseline_mae"]
    forest_value = values["forest_mae"]
    ax.hlines(y, forest_value, baseline, color=color,
              linewidth=2 if label == "Held-out" else 1.2, zorder=1)
    ax.scatter(forest_value, y, color=color, marker="o", s=70, zorder=2)
    ax.scatter(baseline, y, facecolors="white", edgecolors=color, marker="s",
               linewidths=1.5, s=67, zorder=2)
    for x, value in [(forest_value, forest_value), (baseline, baseline)]:
        ax.annotate(f"{value:.2f}", (x, y), xytext=(0, 10), textcoords="offset points",
                    ha="center", color=color)
ax.scatter([], [], color="#245e7d", marker="o", s=58, label="Random forest")
ax.scatter([], [], facecolors="white", edgecolors="#245e7d",
           marker="s", s=58, label="Training mean")
ax.set(xlim=(0, 20), ylim=(-0.55, 1.68), xlabel="Mean absolute error (MPa)")
ax.set_xticks([0, 5, 10, 15, 20])
ax.set_yticks([1, 0], ["Held-out (191 records)", "Validation (208 records)"])
ax.grid(axis="x", color="#dfe6eb")
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.23), ncols=2, frameon=False)
plt.show()

### 次にどの試験記録を集めるか

保留の 191 行を材齢の三範囲に分けます。丸の位置は森に残る誤差，丸と四角を結ぶ線の長さは，同じ行での基準からの改善を表します。残る誤差と改善は，範囲によってそれぞれどう違っていますか。

In [ ]:
age_ranges = [
    ("1–7 days", held_scored["age"].between(1, 7)),
    ("8–28 days", held_scored["age"].between(8, 28)),
    ("29 days or more", held_scored["age"] >= 29),
]
age_summary = pd.DataFrame(
    [
        {
            "range": label,
            "rows": int(mask.sum()),
            "recipes": held_scored.loc[mask, "recipe_id"].nunique(),
            "baseline_mae": held_scored.loc[mask, "baseline_error"].mean(),
            "forest_mae": held_scored.loc[mask, "forest_error"].mean(),
        }
        for label, mask in age_ranges
    ]
).set_index("range")
age_summary["improvement"] = age_summary["baseline_mae"] - age_summary["forest_mae"]
age_summary.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8.1, 4.1), layout="constrained")
ys = range(len(age_summary))
ax.hlines(ys, age_summary["forest_mae"], age_summary["baseline_mae"],
          color="#85949d", linewidth=1.5, zorder=1)
ax.scatter(age_summary["forest_mae"], ys, color="#245e7d",
           marker="o", s=67, label="Random forest", zorder=2)
ax.scatter(age_summary["baseline_mae"], ys, facecolors="white", edgecolors="#245e7d",
           marker="s", s=67, linewidths=1.4, label="Training mean", zorder=2)
for y, (_, row) in enumerate(age_summary.iterrows()):
    for method in ["forest_mae", "baseline_mae"]:
        x = row[method]
        ax.annotate(f"{x:.2f}", (x, y), xytext=(0, 10), textcoords="offset points",
                    ha="center", color="#364653")
ax.set_yticks(list(ys), [f"{label} ({int(row['rows'])} records)"
                        for label, row in age_summary.iterrows()])
ax.set(xlim=(0, 20), ylim=(2.53, -0.68), xlabel="Mean absolute error (MPa)")
ax.set_xticks([0, 5, 10, 15, 20])
ax.grid(axis="x", color="#dfe6eb")
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncols=2, frameon=False)
plt.show()

## 読み手に合わせて報告を直す

**方法を候補に残すか考える人へ。** 学習した森を手順の固定後に保留 191 行・86 配合で確かめた。行を等しく数えた MAE は森 5.55 MPa，学習平均の基準 13.43 MPa だったので，次の確認に森を残す。検証時に見た森の 3.97 MPa より誤差が大きかったため，次は使う予定の試験場所でも配合・材齢・試験条件をそろえて記録し，同じ行で二方法を比較する。

**次の試験記録を集める人へ。** 試験材齢を 29 日以上に指定したときの誤差を次に確かめたい。保留の該当 53 行では森の MAE が 8.37 MPa で，三範囲で最も大きかった。基準の 11.26 MPa より小さいため，次の記録でも両方法を比べる。使う予定の試験場所でどの配合と材齢が指定されるかも確認する。

両方とも同じ計算を使っています。前者は固定後に基準からの改善が残った点を，後者は誤差の残る条件を先に示しました。

## 一文から計算へ戻る

「29 日以上の 53 行で，森の平均絶対誤差は 8.37 MPa」という文を照合します。最初に図と同じ名前で表の行を選びます。次に，保留の評価行のうち材齢が 29 日以上の行を選び，実測から予測を引いて絶対値を求め，平均します。配合数と行数，途中経過に表示する三行と実際に平均する全行を区別してください。

In [ ]:
older_summary = age_summary.loc["29 days or more"]
print(f"Records: {int(older_summary['rows'])}")
print(f"Recipes: {int(older_summary['recipes'])}")
print(f"Forest MAE from the figure's table: {older_summary['forest_mae']:.6f} MPa")

In [ ]:
older_rows = held_scored.loc[held_scored["age"] >= 29]
columns_to_trace = [
    "record_id", "recipe_id", "age", "strength", "forest_prediction", "forest_error",
]
older_rows[columns_to_trace].head(3).round(3)

In [ ]:
older_errors = (older_rows["strength"] - older_rows["forest_prediction"]).abs()
print(f"MAE recomputed from {len(older_errors)} records: {older_errors.mean():.6f} MPa")

行の番号は元表の行を照合するためのものです。教材で，七つの材料量が同じ組合せに同じ配合番号を付けています。強度の単位や資料の利用条件は末尾の出典へ戻って確認します。別の人が計算を最初から再実行するには，どの表とどの列を使うかも伝える必要があります。

## 実習（約 20 分）

### 1．読者を決める（約 4 分）

第 27〜29 章で使った作業ノートを開きます。元の問い，初めの予想，選んだ集計や追加確認を読みます。前の下書きを残しつつ，誰に向けて書くか，その人が次に決めたいことは何かを書きます。

**読者と，次に決めたいこと**　（ここに記入）

### 2．図表を一つ選び，答えを書く（約 6 分）

第 29 章までに出した図表から，読者の問いに答える一つを選びます。中心になる答えを，対象，比較する相手，単位を含む 2〜3 文にしてください。必要な作図や表は自分の作業ノートに残します。

下のセルで，選んだ条件の主な数を再表示できます。前の章で計算した結果がすぐ読める場合は，その結果を使います。

| 前回選んだ問い | セルへ入れる値 |
| --- | --- |
| 材齢範囲の予測 | `question_kind = "prediction"`。前回の `chosen_band` を `"1to7"`，`"8to28"`，`"29plus"` から指定。 |
| 二つの材齢の記述 | `question_kind = "description"`。`chosen_ages` に前回の異なる二材齢（3・7・28・56 から）を入れる。`description_check` は `"median"` または `"signs"`。 |
| 他の既習題材 | `question_kind = "other"`。元の作業ノートの結果を使う。 |

**選んだ図表と，読み手にまず伝える答え**　（ここに記入）

In [ ]:
question_kind = None      # "prediction", "description", or "other"
chosen_band = None        # "1to7", "8to28", or "29plus"
chosen_ages = None        # two distinct ages from 3, 7, 28, 56
description_check = None  # "median" or "signs"

band_labels = {"1to7": "1–7 days", "8to28": "8–28 days", "29plus": "29 days or more"}

if question_kind == "prediction":
    if chosen_band not in band_labels:
        print("前回の材齢範囲を chosen_band に入れてください。")
    else:
        chosen_label = band_labels[chosen_band]
        print("保留の同じ行での二方法の比較（MAE は MPa）")
        display(age_summary.loc[[chosen_label]].round(2))
elif question_kind == "description":
    if (not isinstance(chosen_ages, (list, tuple)) or len(chosen_ages) != 2
            or any(day not in [3, 7, 28, 56] for day in chosen_ages)
            or chosen_ages[0] == chosen_ages[1]):
        print("前回の異なる二材齢を chosen_ages に入れてください。")
    elif description_check not in ["median", "signs"]:
        print("前回の追加確認を description_check に入れてください。")
    else:
        early, late = sorted(chosen_ages)
        records = train.loc[train["age"].isin([early, late])]
        recipe_age = records.groupby(["recipe_id", "age"], as_index=False)["strength"].mean()
        by_recipe = recipe_age.pivot(index="recipe_id", columns="age", values="strength")
        paired = by_recipe.reindex(columns=[early, late]).dropna()
        differences = paired[late] - paired[early]
        print(f"学習表・材齢 {early} 日 → {late} 日: {len(differences)} 配合")
        if description_check == "median":
            display(differences.agg(["mean", "median"]).rename("Difference (MPa)").round(2))
        else:
            import matplotlib.pyplot as plt

            display(pd.Series({
                "Positive": differences.gt(0).sum(),
                "Zero": differences.eq(0).sum(),
                "Negative": differences.lt(0).sum(),
            }, name="Recipes"))
            fig, ax = plt.subplots(figsize=(6.9, 3.55), layout="constrained")
            ax.hist(differences, bins=8, color="#17667f", edgecolor="white")
            ax.axvline(0, color="#555b63", linewidth=1.1, linestyle="--")
            ax.set(
                title=f"{early} to {late} days · {len(differences)} paired recipes",
                xlabel="Later minus earlier strength (MPa)", ylabel="Recipes",
            )
            plt.show()
elif question_kind == "other":
    print("元の作業ノートで，問いと条件をそろえた出力を選んでください。")
else:
    print("前回の結果を使えます。再表示する場合は上の表に沿って指定してください。")

### 3．確認後の説明と，次の比較を書く（約 7 分）

第 29 章の結果を見て，最初の考えのうち何を維持し，どこを変えたかを説明します。予測で選んだ条件なら，基準からの改善と，残る外れを同じ行の結果から読みます。記述で選んだ条件なら，対応した配合にどんな差が見え，平均・中央値や分布が説明に何を加えたかを書きます。

次に調べたいことを一つ挙げ，どの対象から何を記録し，どの条件をそろえて何を比べるかを書きます。

**維持・変更した説明と，次に集める記録**　（ここに記入）

### 4．根拠を照合して保存する（約 3 分）

中心となる一文を一つ選びます。図表で該当する数字を見つけ，集計した行・条件，その値を求めるコード，データの出典の順に戻ります。報告に書いた対象の件数と単位が合うかを確かめます。図表の値を最初から実行して再現するのに必要な資料も分かるようにします。

**照合した一文と，根拠の位置**　（ここに記入）

今回の説明を元の作業ノートへ反映し，下書きと改稿箇所を見比べられる状態で保存します。手法や題材を新たに追加する必要はありません。

## データの出典

I-Cheng Yeh, Concrete Compressive Strength（CC BY 4.0）。元データと配布条件は [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength) を参照してください。教材版の列・配合番号・分割の作り方は [教材データの出典と加工](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/concrete-sources.md) に記載しています。